# 第10章 機械翻訳 
KFTT 日→英。PyTorch `nn.Transformer` をゼロから学習（Mac / MPS）。

In [ ]:
# ===== 先にスレッド関連の環境変数（import より前が必須）=====
import os
# macOS で torch がカーネルごと落ちる典型原因＝OpenMP/MKL のスレッド衝突対策
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # libomp 重複初期化での即クラッシュ回避
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import math, random, time, gc
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.set_num_threads(1)          # PyTorch のスレッドも1に固定
random.seed(42); torch.manual_seed(42)

# まずは CPU で確実に動かす。安定して速度が欲しくなったら False にして MPS を試す
FORCE_CPU = True

if FORCE_CPU:
    device = torch.device("cpu")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("device:", device, "| torch threads:", torch.get_num_threads())


In [ ]:
# ===== 設定 =====
# まず「動く」ことを最優先にした軽量設定。通ったら数値を上げる。
CFG = dict(
    max_len   = 40,
    min_freq  = 2,
    max_vocab = 16000,
    d_model   = 256,
    nhead     = 4,
    num_layers= 3,
    dim_ff    = 512,
    dropout   = 0.1,
    batch_size= 16,
    epochs    = 1,
    lr        = 5e-4,
    train_subset = 20000,   # まず2万文で確認 → 増やす（Noneで全33万文）
)
CFG

## 90. データの準備
`tok/` は分かち書き済み（日=形態素・英=単語）。そのまま語彙化・ID化する。

In [ ]:
# データパス（chapter 11 から chapter 10 のデータを参照）
DATA = os.path.join("kftt-data-1.0", "data", "tok")  # notebook と同じ chapter 11 内
assert os.path.isdir(DATA), f"データが見つからない: {DATA}"

def load(split, lang):
    p = os.path.join(DATA, f"kyoto-{split}.{lang}")
    with open(p, encoding="utf-8") as f:
        return [line.split() for line in f.read().splitlines()]

# train は cln（クリーニング済み）を使う
train_ja = load("train.cln", "ja"); train_en = load("train.cln", "en")
dev_ja   = load("dev", "ja");       dev_en   = load("dev", "en")
test_ja  = load("test", "ja");      test_en  = load("test", "en")
print("raw train pairs:", len(train_ja))

In [ ]:
# 特殊トークン
PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
SPECIALS = [PAD, BOS, EOS, UNK]

def build_vocab(sents, min_freq, max_vocab):
    c = Counter(tok for s in sents for tok in s)
    itos = list(SPECIALS)
    for tok, n in c.most_common():
        if n < min_freq or len(itos) >= max_vocab: break
        itos.append(tok)
    stoi = {t: i for i, t in enumerate(itos)}
    return itos, stoi

ja_itos, ja_stoi = build_vocab(train_ja, CFG["min_freq"], CFG["max_vocab"])
en_itos, en_stoi = build_vocab(train_en, CFG["min_freq"], CFG["max_vocab"])
PAD_ID = ja_stoi[PAD]; BOS_ID = ja_stoi[BOS]; EOS_ID = ja_stoi[EOS]; UNK_ID = ja_stoi[UNK]
print("ja vocab:", len(ja_itos), " en vocab:", len(en_itos))

In [ ]:
# 長すぎる文を除外し、任意で部分集合を取る
def filter_pairs(src, tgt, max_len, subset=None):
    S, T = [], []
    for s, t in zip(src, tgt):
        if 1 <= len(s) <= max_len and 1 <= len(t) <= max_len:
            S.append(s); T.append(t)
    if subset:
        S, T = S[:subset], T[:subset]
    return S, T

train_ja, train_en = filter_pairs(train_ja, train_en, CFG["max_len"], CFG["train_subset"])
print("filtered train pairs:", len(train_ja))

def encode(sent, stoi):
    return [BOS_ID] + [stoi.get(w, UNK_ID) for w in sent] + [EOS_ID]

In [ ]:
class MTDataset(Dataset):
    def __init__(self, src, tgt):
        self.src = [encode(s, ja_stoi) for s in src]
        self.tgt = [encode(t, en_stoi) for t in tgt]
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        return torch.tensor(self.src[i]), torch.tensor(self.tgt[i])

def collate(batch):
    src, tgt = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt = nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=PAD_ID)
    return src, tgt

train_dl = DataLoader(MTDataset(train_ja, train_en), batch_size=CFG["batch_size"],
                      shuffle=True, collate_fn=collate, num_workers=0)
dev_dl   = DataLoader(MTDataset(dev_ja,  dev_en),  batch_size=CFG["batch_size"],
                      shuffle=False, collate_fn=collate, num_workers=0)
print("train batches:", len(train_dl))

## 91. 機械翻訳モデルの訓練

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.src_emb = nn.Embedding(src_vocab, d, padding_idx=PAD_ID)
        self.tgt_emb = nn.Embedding(tgt_vocab, d, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(d, dropout=cfg["dropout"])
        self.transformer = nn.Transformer(
            d_model=d, nhead=cfg["nhead"],
            num_encoder_layers=cfg["num_layers"], num_decoder_layers=cfg["num_layers"],
            dim_feedforward=cfg["dim_ff"], dropout=cfg["dropout"], batch_first=True)
        self.out = nn.Linear(d, tgt_vocab)
        self.d = d
    def encode(self, src, src_pad):
        return self.transformer.encoder(self.pos(self.src_emb(src)*math.sqrt(self.d)),
                                        src_key_padding_mask=src_pad)
    def decode(self, tgt, memory, tgt_mask, tgt_pad, mem_pad):
        return self.transformer.decoder(self.pos(self.tgt_emb(tgt)*math.sqrt(self.d)),
                                        memory, tgt_mask=tgt_mask,
                                        tgt_key_padding_mask=tgt_pad,
                                        memory_key_padding_mask=mem_pad)
    def forward(self, src, tgt):
        src_pad = (src == PAD_ID); tgt_pad = (tgt == PAD_ID)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        mem = self.encode(src, src_pad)
        dec = self.decode(tgt, mem, tgt_mask, tgt_pad, src_pad)
        return self.out(dec)

model = Seq2SeqTransformer(len(ja_itos), len(en_itos), CFG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.1f}M")

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])

def _free():
    gc.collect()
    if device.type == "mps": torch.mps.empty_cache()
    elif device.type == "cuda": torch.cuda.empty_cache()

@torch.no_grad()
def evaluate(dl):
    model.eval(); total, n = 0.0, 0
    for src, tgt in dl:
        src, tgt = src.to(device), tgt.to(device)
        logits = model(src, tgt[:, :-1])
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        total += loss.item(); n += 1
    _free()
    return total / max(n, 1)

def train_epoch(dl):
    model.train(); total, n = 0.0, 0; t0 = time.time()
    for i, (src, tgt) in enumerate(dl):
        src, tgt = src.to(device), tgt.to(device)
        logits = model(src, tgt[:, :-1])
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item(); n += 1
        if i % 200 == 0:
            print(f"  step {i}/{len(dl)} loss {loss.item():.3f} ({time.time()-t0:.0f}s)")
            _free()
    return total / max(n, 1)

In [ ]:
history = []
for ep in range(1, CFG["epochs"]+1):
    tr = train_epoch(train_dl)
    dv = evaluate(dev_dl)
    history.append((tr, dv))
    print(f"epoch {ep}: train {tr:.3f} | dev {dv:.3f} | ppl {math.exp(dv):.1f}")

# 保存
torch.save({"model": model.state_dict(),
            "ja_itos": ja_itos, "en_itos": en_itos, "cfg": CFG},
           "nmt_kftt.pt")
print("saved -> nmt_kftt.pt")

## 92. 機械翻訳モデルの適用

In [ ]:
en_stoi = {t: i for i, t in enumerate(en_itos)}

@torch.no_grad()
def translate(ja_tokens, max_len=100):
    model.eval()
    src = torch.tensor([encode(ja_tokens, ja_stoi)]).to(device)
    mem = model.encode(src, src == PAD_ID)
    ys = torch.tensor([[BOS_ID]]).to(device)
    for _ in range(max_len):
        mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
        dec = model.decode(ys, mem, mask, ys == PAD_ID, src == PAD_ID)
        nxt = model.out(dec[:, -1]).argmax(-1).item()
        ys = torch.cat([ys, torch.tensor([[nxt]]).to(device)], dim=1)
        if nxt == EOS_ID: break
    ids = ys[0].tolist()[1:]
    if EOS_ID in ids: ids = ids[:ids.index(EOS_ID)]
    return [en_itos[i] for i in ids]

# デモ（分かち書き済みの日本語を渡す）
demo = "臨済 宗 は 中国 禅 の ひと つ で ある 。".split()
print("JA:", " ".join(demo))
print("EN:", " ".join(translate(demo)))

In [ ]:
# 未分かち書きの日本語を訳したい場合は MeCab で分割してから translate へ。
# 例（MeCab を入れている場合のみ）:
# import MeCab
# wakati = MeCab.Tagger("-Owakati")
# def translate_raw(text):
#     return " ".join(translate(wakati.parse(text).split()))
# print(translate_raw("臨済宗は中国禅の一つである。"))

## 93. BLEUスコアの計測

In [ ]:
import sacrebleu

@torch.no_grad()
def translate_corpus(src_sents, max_len=100):
    outs = []
    for i, s in enumerate(src_sents):
        outs.append(" ".join(translate(s, max_len)))
        if i % 100 == 0: print(f"  {i}/{len(src_sents)}")
    return outs

# 全 test を訳すと時間がかかる。まず先頭200文で確認 → 全件に変更可
N = 200
hyps = translate_corpus(test_ja[:N])
refs = [" ".join(t) for t in test_en[:N]]

bleu = sacrebleu.corpus_bleu(hyps, [refs], tokenize="none")
print(f"BLEU (first {N}): {bleu.score:.2f}")
print("例 HYP:", hyps[0])
print("例 REF:", refs[0])

## 94. ビーム探索

貪欲デコードをビーム探索に置き換える。ビーム幅を変えながら開発セットの BLEU をプロット。

In [ ]:
@torch.no_grad()
def beam_search(ja_tokens, beam_width=5, max_len=100, length_penalty=0.6):
    """ビーム探索で1文を翻訳。長さ正規化スコアで選択。"""
    model.eval()
    src = torch.tensor([encode(ja_tokens, ja_stoi)]).to(device)
    src_pad = (src == PAD_ID)
    mem = model.encode(src, src_pad)
    beams = [([BOS_ID], 0.0)]        # (トークン列, 対数尤度合計)
    completed = []
    for _ in range(max_len):
        cand = []
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                completed.append((seq, score)); continue
            ys = torch.tensor([seq]).to(device)
            mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
            dec = model.decode(ys, mem, mask, ys == PAD_ID, src_pad)
            logp = torch.log_softmax(model.out(dec[:, -1]), dim=-1)[0]
            topv, topi = logp.topk(beam_width)
            for v, i in zip(topv.tolist(), topi.tolist()):
                cand.append((seq + [i], score + v))
        if not cand:
            break
        norm = lambda x: x[1] / (len(x[0]) ** length_penalty)
        beams = sorted(cand, key=norm, reverse=True)[:beam_width]
        if all(b[0][-1] == EOS_ID for b in beams):
            completed.extend(beams); break
    pool = completed or beams
    best = max(pool, key=lambda x: x[1] / (len(x[0]) ** length_penalty))
    ids = best[0][1:]
    if EOS_ID in ids: ids = ids[:ids.index(EOS_ID)]
    return [en_itos[i] for i in ids]

# デモ：貪欲 vs ビーム
print("greedy:", " ".join(translate(demo)))
print("beam5 :", " ".join(beam_search(demo, beam_width=5)))

In [ ]:
import matplotlib.pyplot as plt

# 開発セットの一部でビーム幅ごとの BLEU を測る（全件は重いので先頭 N_DEV 文）
N_DEV = 100
dev_refs = [" ".join(t) for t in dev_en[:N_DEV]]
widths = [1, 2, 3, 5, 10, 20, 50, 100]
bleus = []
for w in widths:
    hyps = [" ".join(beam_search(dev_ja[i], beam_width=w)) for i in range(N_DEV)]
    s = sacrebleu.corpus_bleu(hyps, [dev_refs], tokenize="none", force=True).score
    bleus.append(s)
    print(f"beam {w:>3}: dev BLEU {s:.2f}")

plt.figure(figsize=(6,4))
plt.plot(widths, bleus, marker="o")
plt.xscale("log"); plt.xlabel("beam width"); plt.ylabel("dev BLEU")
plt.title("94: beam width vs BLEU"); plt.grid(True, alpha=.3)
plt.tight_layout(); plt.savefig("94_beam_bleu.png", dpi=120); plt.show()

## 95. サブワード化

トークン単位を単語/形態素から**サブワード**へ。sentencepiece で日英それぞれの
サブワードモデルを学習し、それで再エンコードして 91–94 を再実行する。

> サブワードは分かち書き前の**生テキスト**(`orig/`)から学習するのが本来。ここでは
> `orig/` を使う。学習後、`USE_SUBWORD=True` にして 91 以降のセルを再実行すれば、
> 語彙・データがサブワードに切り替わる。

In [ ]:
%pip install -q sentencepiece

In [ ]:
import sentencepiece as spm

ORIG = os.path.join("kftt-data-1.0", "data", "orig")
SP_VOCAB = 8000   # サブワード語彙サイズ

# 日英それぞれ sentencepiece モデルを学習（生テキストから）
for lang in ["ja", "en"]:
    model_prefix = f"sp_{lang}"
    if not os.path.exists(model_prefix + ".model"):
        spm.SentencePieceTrainer.train(
            input=os.path.join(ORIG, f"kyoto-train.{lang}"),
            model_prefix=model_prefix, vocab_size=SP_VOCAB,
            character_coverage=0.9995, model_type="unigram",
            pad_id=0, bos_id=1, eos_id=2, unk_id=3,
            input_sentence_size=200000, shuffle_input_sentence=True)
        print(f"trained {model_prefix}.model")
    else:
        print(f"exists  {model_prefix}.model")

sp_ja = spm.SentencePieceProcessor(model_file="sp_ja.model")
sp_en = spm.SentencePieceProcessor(model_file="sp_en.model")
print("sample ja pieces:", sp_ja.encode("臨済宗は中国禅の一つである。", out_type=str)[:12])

In [ ]:
# サブワードに切り替えるスイッチ。True にして 91 以降のデータ準備〜学習セルを再実行する。
USE_SUBWORD = False

if USE_SUBWORD:
    # 生テキストを読み込み直し、sentencepiece でサブワード ID 列にする
    def load_raw(split, lang):
        with open(os.path.join(ORIG, f"kyoto-{split}.{lang}"), encoding="utf-8") as f:
            return f.read().splitlines()
    raw_tr_ja = load_raw("train", "ja"); raw_tr_en = load_raw("train", "en")
    # 語彙は sentencepiece のものを流用
    ja_itos = [sp_ja.id_to_piece(i) for i in range(sp_ja.get_piece_size())]
    en_itos = [sp_en.id_to_piece(i) for i in range(sp_en.get_piece_size())]
    ja_stoi = {t: i for i, t in enumerate(ja_itos)}
    en_stoi = {t: i for i, t in enumerate(en_itos)}
    PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

    def encode(sent, sp):
        # sent は「生テキスト文字列」。sp は sp_ja / sp_en
        ids = sp.encode(sent, out_type=int)
        return [BOS_ID] + ids + [EOS_ID]

    # 注意：この後は MTDataset / DataLoader / model 構築 / 学習(91)以降を
    #        再実行すること。translate も en_itos/ja_stoi を参照するので整合する。
    print("サブワード語彙:", len(ja_itos), len(en_itos), " → 91以降を再実行してください")
else:
    print("USE_SUBWORD=False（単語/形態素のまま）。サブワードで実験するなら True にして91以降を再実行。")

## 96. 学習過程の可視化 (TensorBoard)

学習/開発の損失と BLEU を TensorBoard に記録する強化版の学習ループ。
実行後、`%tensorboard --logdir runs` でグラフを表示。

In [ ]:
from torch.utils.tensorboard import SummaryWriter

@torch.no_grad()
def dev_bleu(n=100):
    hyps = [" ".join(translate(dev_ja[i])) for i in range(n)]
    refs = [" ".join(t) for t in dev_en[:n]]
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="none", force=True).score

def train_with_tb(epochs=None, logdir="runs/kftt"):
    epochs = epochs or CFG["epochs"]
    writer = SummaryWriter(logdir)
    step = 0
    for ep in range(1, epochs+1):
        model.train()
        for src, tgt in train_dl:
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src, tgt[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            if step % 100 == 0:
                writer.add_scalar("train/loss", loss.item(), step)
            step += 1
        dl_loss = evaluate(dev_dl)
        db = dev_bleu(100)
        writer.add_scalar("dev/loss", dl_loss, ep)
        writer.add_scalar("dev/bleu", db, ep)
        writer.add_scalar("dev/ppl", math.exp(dl_loss), ep)
        print(f"epoch {ep}: dev loss {dl_loss:.3f} | dev BLEU {db:.2f}")
    writer.close()
    print("logged -> ", logdir)

# 実行（TensorBoard用のログを runs/ に書き出す）
train_with_tb()

In [ ]:
# ノートブック内で TensorBoard を開く（別途 `tensorboard --logdir runs` でも可）
%load_ext tensorboard
%tensorboard --logdir runs

## 97. ハイパーパラメータの調整

いくつかの設定で小さく学習し、開発 BLEU が最大のものを探す。
（フル探索は重いので、各候補は少数エポック・小サブセットで比較する簡易版。）

In [ ]:
def build_and_train(cfg, n_epochs=1, subset=10000):
    """与えられた cfg で新しいモデルを作り軽く学習し、dev BLEU を返す。"""
    global model   # translate() が参照するグローバル model を差し替える
    # データ（サブセット）を作り直す
    s_ja, s_en = filter_pairs(*(train_ja, train_en), cfg["max_len"], subset)         if False else (train_ja[:subset], train_en[:subset])
    dl = DataLoader(MTDataset(s_ja, s_en), batch_size=cfg["batch_size"],
                    shuffle=True, collate_fn=collate, num_workers=0)
    m = Seq2SeqTransformer(len(ja_itos), len(en_itos), cfg).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=cfg["lr"])
    crit = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    for _ in range(n_epochs):
        m.train()
        for src, tgt in dl:
            src, tgt = src.to(device), tgt.to(device)
            logits = m(src, tgt[:, :-1])
            loss = crit(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
    # このモデルで dev BLEU（translate はグローバル model を使うので差し替え）
    model = m
    return dev_bleu(100)

# 探索する候補（小さめ）
grid = [
    {**CFG, "d_model":128, "num_layers":2, "lr":5e-4},
    {**CFG, "d_model":256, "num_layers":3, "lr":5e-4},
    {**CFG, "d_model":256, "num_layers":3, "lr":1e-3},
]
results = []
for i, g in enumerate(grid):
    b = build_and_train(g, n_epochs=1, subset=10000)
    results.append((b, g))
    print(f"[{i}] d={g['d_model']} L={g['num_layers']} lr={g['lr']} -> dev BLEU {b:.2f}")

best_bleu, best_cfg = max(results, key=lambda x: x[0])
print("\nBEST dev BLEU:", round(best_bleu,2),
      "| d_model", best_cfg["d_model"], "layers", best_cfg["num_layers"], "lr", best_cfg["lr"])

## 98. ドメイン適応（ローカル降級版・実行可能）

本来は JESC / JParaCrawl（外部・数百万〜数千万文）で**事前学習**し、KFTT で**微調整**する。
だが外部データ＋GPU が必要でローカル Mac では非現実的。

そこでここでは、**91 で保存した `nmt_kftt.pt` を「事前学習済みモデル」に見立て**、
それを KFTT で追加学習（微調整）するパイプラインを**実際に最後まで動かす**。
これでドメイン適応のコード経路（読み込み→低学習率で微調整→評価）がローカルで通る。

> 本物の 98 にするには、この `nmt_kftt.pt` を JESC 等で学習した汎用モデルに差し替えるだけ。
> その手順は下の注釈に残してある。

In [ ]:
# ---- 98 ローカル降級版：ドメイン適応パイプラインを実際に動かす ----

def finetune_from(pretrained_path, adapt_dl, lr=1e-4, epochs=2):
    """事前学習済みチェックポイントを読み込み、低い学習率で追加学習(微調整)する。"""
    ckpt = torch.load(pretrained_path, map_location=device)
    m = Seq2SeqTransformer(len(ckpt["ja_itos"]), len(ckpt["en_itos"]), ckpt["cfg"]).to(device)
    m.load_state_dict(ckpt["model"])
    opt = torch.optim.AdamW(m.parameters(), lr=lr)       # 低LRで微調整
    crit = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    for ep in range(epochs):
        m.train(); last = 0.0
        for src, tgt in adapt_dl:
            src, tgt = src.to(device), tgt.to(device)
            logits = m(src, tgt[:, :-1])
            loss = crit(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); last = loss.item()
        print(f"  finetune epoch {ep+1}/{epochs}: last loss {last:.3f}")
    return m

def test_bleu(m, n=200):
    """モデル m で KFTT test の先頭 n 文を訳し BLEU を返す（translate はグローバル model を使う）。"""
    global model
    model = m
    hyps = [" ".join(translate(test_ja[i])) for i in range(n)]
    refs = [" ".join(t) for t in test_en[:n]]
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="none", force=True).score

# --- 実行：91の nmt_kftt.pt を「事前学習済み」とみなして微調整する ---
print("微調整前 test BLEU:", round(test_bleu(model, 200), 2))
adapted = finetune_from("nmt_kftt.pt", train_dl, lr=1e-4, epochs=2)
print("微調整後 test BLEU:", round(test_bleu(adapted, 200), 2))
torch.save({"model": adapted.state_dict(), "ja_itos": ja_itos,
            "en_itos": en_itos, "cfg": CFG}, "nmt_kftt_adapted.pt")

# === 本物のドメイン適応にする場合（GPU環境） ===
# 1) JESC/JParaCrawl を取得（下記URL）
#    JESC:       https://nlp.stanford.edu/projects/jesc/
#    JParaCrawl: http://www.kecl.ntt.co.jp/icl/lirg/jparacrawl/
# 2) 95のサブワードで KFTT と統一エンコードし、外部データで汎用モデルを事前学習 → general.pt
# 3) finetune_from("general.pt", train_dl, lr=1e-4, epochs=3) で KFTT に適応
# 4) test_bleu で KFTTのみ学習(93)と比較 → 通常は向上する

## 99. 翻訳サーバの構築

Web ブラウザ上で日本語を入力→英訳を表示するデモ。Gradio で簡単に立てる。
入力は**分かち書き済み**(スペース区切り)を想定。MeCab があれば生文にも対応。

In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr

# MeCab があれば生の日本語も分かち書きする（無ければスペース区切り入力を想定）
try:
    import MeCab
    _wakati = MeCab.Tagger("-Owakati")
    def _split_ja(text): return _wakati.parse(text).split()
    _has_mecab = True
except Exception:
    def _split_ja(text): return text.split()
    _has_mecab = False

def translate_ui(text, use_beam=True, beam_width=5):
    toks = _split_ja(text)
    if not toks:
        return ""
    out = beam_search(toks, beam_width=int(beam_width)) if use_beam else translate(toks)
    return " ".join(out)

demo = gr.Interface(
    fn=translate_ui,
    inputs=[gr.Textbox(label="日本語" + ("（生文OK）" if _has_mecab else "（分かち書きして入力）")),
            gr.Checkbox(value=True, label="ビーム探索を使う"),
            gr.Slider(1, 20, value=5, step=1, label="ビーム幅")],
    outputs=gr.Textbox(label="English"),
    title="KFTT 日→英 ニューラル機械翻訳",
    examples=[["臨済 宗 は 中国 禅 の ひと つ で ある 。", True, 5]],
)
demo.launch()   # ブラウザでローカルURL(http://127.0.0.1:7860)が開く